# 🩸 MedSynapse — Type-2 Diabetes Mellitus Clinical Risk Diagnostic System
### Optimized Ensemble Soft-Voting Classifier on Glycemic, Anthropometric & Metabolic Biomarkers

[![Dataset: Diabetes](https://img.shields.io/badge/Dataset-Pima%20Indians%20Diabetes-blue?style=for-the-badge&logo=kaggle)](https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database)
[![Architecture: Ensemble Voting](https://img.shields.io/badge/Architecture-Soft--Voting%20Ensemble-green?style=for-the-badge&logo=scikit-learn)](https://scikit-learn.org)
[![Accuracy: 74.68%](https://img.shields.io/badge/Test%20Accuracy-74.68%25-brightgreen?style=for-the-badge)]()
[![Classes: 2 Diagnostic States](https://img.shields.io/badge/Classes-Diabetic%20(1)%20%7C%20Non--Diabetic%20(0)-orange?style=for-the-badge)]()
[![Platform: MedSynapse](https://img.shields.io/badge/Platform-MedSynapse%20AI-red?style=for-the-badge)]()

---

### 📋 Overview & Clinical Significance
Type-2 Diabetes Mellitus is a chronic metabolic disorder resulting from progressive insulin secretion defects and peripheral insulin resistance. Early clinical risk stratification allows lifestyle intervention before microvascular and macrovascular complications develop. This diagnostic pipeline synthesizes 8 metabolic and demographic biomarkers to predict diabetic risk:

| Diagnostic State | Clinical Status | Metabolic Presentation & Diagnostic Criteria |
|:---|:---|:---|
| **`Diabetic` (1)** | **Impaired Glucose Homeostasis** | Fasting glucose ≥126 mg/dL, 2-hr OGTT ≥200 mg/dL, elevated insulin resistance or BMI. |
| **`Non-Diabetic` (0)** | **Euglycemic Control** | Normal fasting glucose (<100 mg/dL), preserved physiological insulin sensitivity. |

---

### 🎯 Pipeline Architecture
1. **Environment Setup & Data Ingestion**: Automated search across Kaggle mount points and dataset loading.
2. **Clinical Preprocessing & Imputation**: Physiological zero-replacement (insulin, skin thickness, BMI) and StandardScaler normalization.
3. **Multi-Model Soft-Voting Ensemble**: Random Forest, Gradient Boosting, Extra Trees, and L2-regularized Logistic Regression.
4. **Diagnostic Evaluation**: Accuracy calculation, per-class classification report, and confusion matrix.
5. **Production Packaging**: Bundling model, scaler, and metadata into `diabetes_artifacts.zip` for MedSynapse.


In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import pickle
import warnings
warnings.filterwarnings('ignore')

## 1. 📂 Dataset Ingestion & Physiological Imputation Pipeline
Resolves dataset location, imputes non-physiological zero values with feature medians, and scales features via `StandardScaler`.


In [7]:
# ====================================================================
# 🌐 DATASET AUTO-RESOLVER & KAGGLE CONNECTOR (Diabetes Mellitus)
# ====================================================================
import os
import urllib.request
import pandas as pd
import numpy as np

candidate_files = [
    '../datasets/diabetes.csv',
    './datasets/diabetes.csv',
    './diabetes.csv',
    '/kaggle/input/pima-indians-diabetes-database/diabetes.csv',
    '/kaggle/input/diabetes-dataset/diabetes.csv'
]

csv_path = None
for f in candidate_files:
    if os.path.exists(f):
        csv_path = f
        print(f"✅ Found dataset file at: {csv_path}")
        break

if not csv_path:
    print("⚡ Dataset not found locally. Attempting KaggleHub download...")
    try:
        import kagglehub
        download_dir = kagglehub.dataset_download("uciml/pima-indians-diabetes-database")
        for root, _, files in os.walk(download_dir):
            for file in files:
                if file.endswith('.csv'):
                    csv_path = os.path.join(root, file)
                    print(f"✅ Downloaded via KaggleHub: {csv_path}")
                    break
    except Exception as e:
        print(f"⚠️ KaggleHub notice: {e}. Falling back to online mirror...")
        fallback_url = "https://raw.githubusercontent.com/plotly/datasets/master/diabetes.csv"
        os.makedirs("datasets", exist_ok=True)
        csv_path = "datasets/diabetes.csv"
        urllib.request.urlretrieve(fallback_url, csv_path)
        print(f"✅ Downloaded mirror to: {csv_path}")

df = pd.read_csv(csv_path)
print(f"📊 Loaded Dataset Shape: {df.shape}")

# 1. Handle 0 values (Replace with NaN)
cols_to_fix = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_to_fix] = df[cols_to_fix].replace(0, np.nan)

# 2. KNN Imputation (More robust than median)
knn_imputer = KNNImputer(n_neighbors=5)
df[cols_to_fix] = knn_imputer.fit_transform(df[cols_to_fix])

# 3. Outlier Capping (Winsorization)
def cap_outliers(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    low_lim = Q1 - 1.5 * IQR
    up_lim = Q3 + 1.5 * IQR
    df[col] = np.where(df[col] < low_lim, low_lim, df[col])
    df[col] = np.where(df[col] > up_lim, up_lim, df[col])
    return df

for col in ['Insulin', 'DiabetesPedigreeFunction']:
    df = cap_outliers(df, col)

# 4. Feature Engineering: BMI Categories
df['BMI_Cat'] = pd.cut(df['BMI'], bins=[0, 18.5, 24.9, 29.9, 100], labels=[0, 1, 2, 3]).astype(int)

# 5. Split Features & Target
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# 6. Standardization
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)


✅ Found dataset file at: ./datasets/diabetes.csv
📊 Loaded Dataset Shape: (768, 9)


## 2. 🤝 Multi-Model Soft-Voting Ensemble Architecture
Constructs soft-voting ensemble pooling probabilistic predictions from 4 diverse machine learning estimators.


In [8]:
# Base Models
rf = RandomForestClassifier(n_estimators=20, max_depth=20, min_samples_split=5, min_samples_leaf=1, random_state=42)
gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
lr = LogisticRegression(solver='liblinear', random_state=42)

# Voting Classifier (Soft Voting for probability averaging)
voting_clf = VotingClassifier(
    estimators=[
        ('rf', rf),
        ('gb', gb),
        ('lr', lr)
    ],
    voting='soft'
)

voting_clf.fit(X_train, y_train)

VotingClassifier(estimators=[('rf',
                              RandomForestClassifier(max_depth=20,
                                                     min_samples_split=5,
                                                     n_estimators=20,
                                                     random_state=42)),
                             ('gb',
                              GradientBoostingClassifier(random_state=42)),
                             ('lr',
                              LogisticRegression(random_state=42,
                                                 solver='liblinear'))],
                 voting='soft')

## 3. 📊 Clinical Evaluation & Diagnostic Performance
Computes test set accuracy (74.68%), precision, recall, and confusion matrix.


In [9]:
y_train_pred = voting_clf.predict(X_train)
y_test_pred = voting_clf.predict(X_test)

train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)

print(f"Training Accuracy: {train_acc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

print("\nClassification Report (Test):")
print(classification_report(y_test, y_test_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

Training Accuracy: 0.9235
Test Accuracy: 0.7468

Classification Report (Test):
              precision    recall  f1-score   support

           0       0.79      0.83      0.81       100
           1       0.65      0.59      0.62        54

    accuracy                           0.75       154
   macro avg       0.72      0.71      0.72       154
weighted avg       0.74      0.75      0.74       154


Confusion Matrix:
[[83 17]
 [22 32]]


## 4. 📦 Export Model & Preprocessing Transformers
Serializes voting classifier and fitted scaler using Pickle.


In [10]:
# ====================================================================
# 💾 SAVE TRAINED MODEL & ARTIFACTS
# ====================================================================
import os
import pickle

# Ensure local and repo model directories exist
os.makedirs('../models/Diabetes_model', exist_ok=True)
os.makedirs('../models', exist_ok=True)

# 1. Save Primary Model and Scaler to working directory
pickle.dump(voting_clf, open('diabetes_model.pkl', 'wb'))
pickle.dump(scaler, open('diabetes_scaler.pkl', 'wb'))

# 2. Also mirror to repo models directory if accessible
try:
    pickle.dump(voting_clf, open('../models/Diabetes_model/diabetes_model.pkl', 'wb'))
    pickle.dump(scaler, open('../models/Diabetes_model/diabetes_scaler.pkl', 'wb'))
    pickle.dump(voting_clf, open('../models/diabetes_model.pkl', 'wb'))
    pickle.dump(scaler, open('../models/diabetes_scaler.pkl', 'wb'))
except Exception as e:
    pass

print('✅ Optimized Voting Classifier saved as diabetes_model.pkl and diabetes_scaler.pkl')


✅ Optimized Voting Classifier saved as diabetes_model.pkl and diabetes_scaler.pkl


## 5. 🔍 Model Verification & Sample Prediction
Loads saved artifacts and verifies predictive pipeline on sample patient records.


In [11]:
# Verification
import pickle
import numpy as np

# Load model & scaler
model = pickle.load(open('diabetes_model.pkl', 'rb'))
scaler_loaded = pickle.load(open('diabetes_scaler.pkl', 'rb'))

try:
    sample_data = X_test[0:1]
    prediction = model.predict(sample_data)
    prob = model.predict_proba(sample_data)
    print(f'Predicted Class: {prediction[0]} (Probability: {prob[0][prediction[0]]:.4f})')
    print(f'Actual Class: {y_test.iloc[0] if hasattr(y_test, "iloc") else y_test[0]}')
except Exception as e:
    print(f'Verification notice: {e}')


Predicted Class: 1 (Probability: 0.6461)
Actual Class: 0


## 6. 📦 Production Packaging & MedSynapse Artifact Export
Packages model, scaler, and diagnostic metadata into `diabetes_artifacts.zip` for MedSynapse.


In [12]:
# ====================================================================
# 📥 SAVE & DOWNLOAD TRAINED ARTIFACTS ON KAGGLE
# ====================================================================
import os
import zipfile
from IPython.display import FileLink, display

saved_files = ['diabetes_model.pkl', 'diabetes_scaler.pkl']
saved_files = [f for f in saved_files if os.path.exists(f)]

zip_filename = 'diabetes_artifacts.zip'
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for file in saved_files:
        zipf.write(file)
        print(f'📦 Packaged in zip: {file}')

print('')
print('' + f'✅ Bundle ready: {zip_filename} ({os.path.getsize(zip_filename)} bytes)')
print('👉 Click the link below to download your trained artifacts:')
display(FileLink(zip_filename))


: 